Goal is identify why a specific CV-fold is an outlier compared to others.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display_html
import seaborn as sns

from pygmt_helper import plotting

import nn_gmm as nng
import ml_tools as mlt

PLOT_IMS = ["pSA_0.01", "pSA_0.1", "pSA_0.5", "pSA_1.0", "pSA_3.0", "pSA_10.0"]

In [ ]:
results_dir = Path("/Users/claudy/dev/work/data/nn_gmm/results/0929_1956_cv_v3_locSiteCond_25Epochs_3E4Folds_saveSampleWeights")
cv_fold = 9

In [ ]:
run_config = nng.RunConfig.from_yaml(results_dir / "run_config.yaml")

In [ ]:
val_df = pd.read_parquet(results_dir / "val_results.parquet")
val_df = val_df[val_df.cv_iter == cv_fold].sort_index()
val_df.loc[:, run_config.pred_mean_keys] = np.exp(val_df[run_config.pred_mean_keys])

In [ ]:
with nng.IMDB(run_config.imdb_ffp, readonly=True) as imdb:
    event_df = imdb.get_event_df()
    site_df = imdb.get_site_df()
    sim_df = imdb.get_im_data(run_config.ims, val_df.index.values).sort_index()
    record_info_df = imdb.get_record_info_df(record_int_ids=val_df.index.values).sort_index()
    record_info_df["site_event_int_id"] = nng.utils.get_site_event_int_id(record_info_df.site_int_id.values, 
                                                                          record_info_df.event_int_id.values)
    event_site_df = imdb.get_site_event_df(site_event_int_ids=record_info_df.site_event_int_id.values)
    rel_df = imdb.get_rel_df()
assert sim_df.index.equals(val_df.index)
assert record_info_df.index.equals(val_df.index)

In [ ]:
sim_df["event_id"] = record_info_df["event_id"]
sim_df["rel_id"] = record_info_df["rel_id"]
sim_df["site_id"] = record_info_df["site_id"]
sim_df["site_int_id"] = record_info_df["site_int_id"]

In [ ]:
val_df["rel_id"] = record_info_df["rel_id"]
val_df["site_id"] = record_info_df["site_id"]
val_df["site_int_id"] = record_info_df["site_int_id"]
val_df["event_id"] = record_info_df["event_id"]
val_df["event_int_id"] = record_info_df["event_int_id"]

In [ ]:
loss_columns = [col for col in val_df.columns if col.endswith("_loss")]
loss_df = val_df[loss_columns].astype(np.float64).rename(columns=lambda x: x.replace("_loss", ""))
record_loss = loss_df.mean(axis=1).to_frame("mean_loss")
record_loss["site_id"] = record_info_df.loc[record_loss.index, "site_id"]
record_loss["site_int_id"] = record_info_df.loc[record_loss.index, "site_int_id"]
record_loss["event_id"] = record_info_df.loc[record_loss.index, "event_id"]
record_loss["rel_id"] = record_info_df.loc[record_loss.index, "rel_id"]
record_loss["rrup"] = event_site_df.loc[record_info_df.site_event_int_id.values, "rrup"].values
record_loss["lon"] = site_df.loc[record_info_df.site_int_id.values, "lon"].values
record_loss["lat"] = site_df.loc[record_info_df.site_int_id.values, "lat"].values
record_loss["rel_int_id"] = record_info_df.loc[record_loss.index, "rel_int_id"]

In [ ]:
loss_df["sample_weight"] = val_df["sample_weight"]
loss_df["rel_id"] = record_info_df.loc[loss_df.index, "rel_id"]
record_loss["sample_weight"] = loss_df["sample_weight"]

In [ ]:
pred_cols = [col for col in val_df.columns if col.endswith("_pred")]
pred_std_cols = [col for col in val_df.columns if col.endswith("_pred_std")]

## Loss Distribution

In [ ]:
## Loss distribution
fig, ax = plt.subplots(figsize=(16, 6))

ax.hist(record_loss.mean_loss, bins=100)
ax.set_yscale("log")

# ax.axvline(record_loss.mean_loss.mean(), color="red", linestyle="--", label="Mean")
median = record_loss.mean_loss.median()
ax.axvline(median, color="black", label=f"Median - {median:.2f}")

qt_16, qt_84 = record_loss.mean_loss.quantile([0.16, 0.84])
ax.axvline(qt_16, color="green", label=f"16th percentile - {qt_16:.2f}")
ax.axvline(qt_84, color="green", label=f"84th percentile - {qt_84:.2f}")

qt_99 = record_loss.mean_loss.quantile(0.99)
ax.axvline(qt_99, color="orange", label=f"99th percentile - {qt_99:.2f}")

qt_999 = record_loss.mean_loss.quantile(0.999)
ax.axvline(qt_999, color="red", label=f"99.9th percentile - {qt_999:.2f}")

ax.set_xlim(record_loss.mean_loss.min(), record_loss.mean_loss.max())
ax.legend()

fig.tight_layout()

### IM Loss Distribution

In [ ]:
im_median_loss = loss_df[run_config.ims].median(axis=0)
im_qt_16 = loss_df[run_config.ims].quantile(0.16, axis=0)
im_qt_84 = loss_df[run_config.ims].quantile(0.84, axis=0)
im_qt_99 = loss_df[run_config.ims].quantile(0.99, axis=0)
im_qt_0p01 = loss_df[run_config.ims].quantile(0.0001, axis=0)
im_qt_999 = loss_df[run_config.ims].quantile(0.999, axis=0)
im_qt_0p001 = loss_df[run_config.ims].quantile(0.0001, axis=0)

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

ax.plot(nng.constants.PSA_PERIODS, im_median_loss[nng.constants.PSA_KEYS], label="Median", color="blue")

ax.fill_between(nng.constants.PSA_PERIODS, im_qt_16[nng.constants.PSA_KEYS], im_qt_84[nng.constants.PSA_KEYS], 
                color="lightblue", alpha=0.5)
ax.plot(nng.constants.PSA_PERIODS, im_qt_16[nng.constants.PSA_KEYS], color="blue", linestyle="--", label="16th/84th percentile")
ax.plot(nng.constants.PSA_PERIODS, im_qt_84[nng.constants.PSA_KEYS], color="blue", linestyle="--")

ax.plot(nng.constants.PSA_PERIODS, im_qt_99[nng.constants.PSA_KEYS], color="orange", linestyle="--", label="0.01th/99th percentile")
ax.plot(nng.constants.PSA_PERIODS, im_qt_0p01[nng.constants.PSA_KEYS], color="orange", linestyle="--")
ax.plot(nng.constants.PSA_PERIODS, im_qt_999[nng.constants.PSA_KEYS], color="red", linestyle="--", label="0.001th/99.9th percentile")
ax.plot(nng.constants.PSA_PERIODS, im_qt_0p001[nng.constants.PSA_KEYS], color="red", linestyle="--")
ax.legend()

ax.set_xscale("log")
ax.set_xlim(0.01, 10)
    
fig.tight_layout()

In [ ]:
fig, axs = mlt.plotting.get_fig_axes(len(PLOT_IMS), 2, -1, ind_figsize=(8, 6))

for i, (ax, cur_im) in enumerate(zip(axs, PLOT_IMS)):
    ax.hist(loss_df[cur_im], bins=100)

    ax.axvline(im_median_loss[cur_im], color="black", label="Median")
    ax.axvline(im_qt_16[cur_im], color="green", label="16th/84th percentile")
    ax.axvline(im_qt_84[cur_im], color="green")
    ax.axvline(im_qt_99[cur_im], color="orange", label="99th percentile")
    ax.axvline(im_qt_999[cur_im], color="red", label="99.9th percentile")

    ax.set_xlim(loss_df[run_config.ims].min().min(), loss_df[run_config.ims].max().max())

    if i == 0:
        ax.legend()

    ax.text(0.5, 0.95, cur_im, transform=ax.transAxes, horizontalalignment="center", verticalalignment="center", fontsize=14)

    ax.set_yscale("log")

fig.tight_layout()

## Bad Record Investigation

In [ ]:
record_loss_threshold = record_loss.mean_loss.quantile(0.999)
print("Record loss threshold:", record_loss_threshold)

In [ ]:
mask = record_loss.mean_loss > record_loss_threshold
bad_records = record_loss[mask].sort_values("mean_loss", ascending=False)
bad_events = bad_records.event_id.unique().tolist()
print("Number of bad records:", len(bad_records))
print("Events with bad records:", bad_events)
print("Number of sites with bad records:", bad_records.site_id.nunique())

In [ ]:
event_info_group = bad_records.groupby(["event_id", "rel_id"], observed=True)
event_rel_info = event_info_group["mean_loss"].mean().to_frame("mean_site_loss")
event_rel_info["n_sites"] = event_info_group["site_id"].nunique()

In [ ]:
event_rel_info.sort_values("n_sites", ascending=False)

### Spatial - Sites and Faults

In [ ]:
spatial_plot = nng.plots_spatial.SpatialPlot(plot_topo=False).plot_faults(bad_events, pen="0.75p,red").plot_sites(bad_records.groupby("site_id").first(), style="t0.1c", fill="blue")
spatial_plot.fig.show(dpi=900, width=1200)

## Realisation Investigation

In [ ]:
cur_event_id = "AlpineF2K"
cur_rel_id = "AlpineF2K_REL08"

cur_im = PLOT_IMS[0]
cur_sim_im_df = sim_df.loc[sim_df.rel_id == cur_rel_id]
cur_pred_im_df = val_df.loc[val_df.rel_id == cur_rel_id]

cur_sim_im_df.loc[:,["lon", "lat"]] = site_df.loc[cur_sim_im_df.site_int_id.values, ["lon", "lat"]].values
cur_pred_im_df.loc[:,["lon", "lat"]] = site_df.loc[cur_pred_im_df.site_int_id.values, ["lon", "lat"]].values


In [ ]:
cur_record_loss = record_loss.loc[record_loss.rel_id == cur_rel_id]

### Spatial IM Comparison

In [ ]:
cur_rel_info = rel_df.loc[rel_df.rel_id == cur_rel_id].squeeze()

In [ ]:
loss_plot = nng.plots_spatial.SpatialPlot(plot_topo=False)

loss_grid = plotting.create_grid(cur_record_loss, "mean_loss", grid_spacing="500e/500e")
plotting.plot_grid(
        loss_plot.fig,
        loss_grid,
        cb_label="Mean Record Loss",
        plot_contours=False,
        reverse_cmap=True,
        cmap="hot",
        cmap_limits=(0.0, 100, 10),
        cmap_limit_colors=("white", "black"),
)

loss_plot.plot_sites(cur_sim_im_df).plot_sites(cur_sim_im_df.loc[cur_sim_im_df.index.isin(bad_records.index)], style="t0.1c", fill="blue").plot_faults([cur_event_id], pen="0.75p,red")


sample_weight_plot = nng.plots_spatial.SpatialPlot(plot_topo=False)

sample_weight_grid = plotting.create_grid(cur_record_loss, "sample_weight", grid_spacing="500e/500e")
plotting.plot_grid(
        sample_weight_plot.fig,
        sample_weight_grid,
        cb_label="Sample Weight",
        plot_contours=False,
        cmap="hot",
        cmap_limits=(1.0, 5.0, 0.5),
        cmap_limit_colors=("white", "black"),
        reverse_cmap=True
)

sample_weight_plot.plot_sites(cur_sim_im_df).plot_sites(cur_sim_im_df.loc[cur_sim_im_df.index.isin(bad_records.index)], style="t0.1c", fill="blue").plot_faults([cur_event_id], pen="0.75p,red")

fig = nng.plot_utils.pygmt_side_by_side(loss_plot.fig, sample_weight_plot.fig, dpi=300, title_1="Loss", title_2="Sample Weight")
fig.tight_layout()
plt.show(fig)

#### pSA(0.01)

In [ ]:
sim_plot = (nng.plots_spatial.SpatialPlot(plot_topo=False)
            .plot_im_values(cur_sim_im_df, cur_im)
            .plot_sites(cur_sim_im_df)
            .plot_sites(cur_sim_im_df.loc[cur_sim_im_df.index.isin(bad_records.index)], style="t0.1c", fill="blue")
            .plot_faults([cur_event_id], pen="0.75p,red")
            .plot_hypocentre(cur_rel_info.hypo_lon, cur_rel_info.hypo_lat)
)

pred_plot = (nng.plots_spatial.SpatialPlot(plot_topo=False)
        .plot_im_values(cur_pred_im_df, f"{cur_im}_pred", cur_im)
        .plot_sites(cur_sim_im_df)
        .plot_sites(cur_sim_im_df.loc[cur_sim_im_df.index.isin(bad_records.index)], style="t0.1c", fill="blue")
        .plot_faults([cur_event_id], pen="0.75p,red")
        .plot_hypocentre(cur_rel_info.hypo_lon, cur_rel_info.hypo_lat)
)

fig = nng.plot_utils.pygmt_side_by_side(sim_plot.fig, pred_plot.fig, dpi=300, title_1="Simulation", title_2="Prediction")  
fig.tight_layout()
plt.show(fig)

In [ ]:
cur_res_df = (np.log(cur_sim_im_df.loc[cur_pred_im_df.index, cur_im]) - np.log(cur_pred_im_df[f"{cur_im}_pred"])).to_frame("residual")
cur_res_df.loc[:, ["lon", "lat"]] = cur_sim_im_df.loc[cur_res_df.index, ["lon", "lat"]]

In [ ]:
pred_std_plot = (nng.plots_spatial.SpatialPlot(plot_topo=False)
        .plot_im_values(cur_pred_im_df, f"{cur_im}_pred_std", cur_im)
        .plot_sites(cur_sim_im_df)
        .plot_sites(cur_sim_im_df.loc[cur_sim_im_df.index.isin(bad_records.index)], style="t0.1c", fill="blue")
        .plot_faults([cur_event_id], pen="0.75p,red")
        .plot_hypocentre(cur_rel_info.hypo_lon, cur_rel_info.hypo_lat)
)

res_plot = (nng.plots_spatial.SpatialPlot(plot_topo=False)
        .plot_ratio(cur_res_df, "residual", cb_label=f"{cur_im}")
        .plot_sites(cur_sim_im_df)
        .plot_sites(cur_sim_im_df.loc[cur_sim_im_df.index.isin(bad_records.index)], style="t0.1c", fill="blue")
        .plot_faults([cur_event_id], pen="0.75p,red")
        .plot_hypocentre(cur_rel_info.hypo_lon, cur_rel_info.hypo_lat)
)

fig = nng.plot_utils.pygmt_side_by_side(pred_std_plot.fig, res_plot.fig, dpi=300, title_1="Predicted Standard Deviation", title_2="Residuals")  
fig.tight_layout()
plt.show(fig)

#### IM Residuals

In [ ]:
n_ims = len(PLOT_IMS)

for i in range(n_ims // 2):
    im1 = PLOT_IMS[2 * i]
    im2 = PLOT_IMS[2 * i + 1]

    cur_res_df = (np.log(cur_sim_im_df.loc[cur_pred_im_df.index, im1]) - np.log(cur_pred_im_df[f"{im1}_pred"])).to_frame("residual")
    cur_res_df.loc[:, ["lon", "lat"]] = cur_sim_im_df.loc[cur_res_df.index, ["lon", "lat"]]
    res_1_plot = (nng.plots_spatial.SpatialPlot(plot_topo=False)
            .plot_ratio(cur_res_df, "residual", cb_label=f"{im1}", cmap_limits=(-1.0, 1.0, 0.1))
            .plot_sites(cur_sim_im_df)
            .plot_sites(cur_sim_im_df.loc[cur_sim_im_df.index.isin(bad_records.index)], style="t0.1c", fill="blue")
            .plot_faults([cur_event_id], pen="0.75p,red")
            .plot_hypocentre(cur_rel_info.hypo_lon, cur_rel_info.hypo_lat)
    )

    cur_res_df = (np.log(cur_sim_im_df.loc[cur_pred_im_df.index, im2]) - np.log(cur_pred_im_df[f"{im2}_pred"])).to_frame("residual")
    cur_res_df.loc[:, ["lon", "lat"]] = cur_sim_im_df.loc[cur_res_df.index, ["lon", "lat"]]
    res_2_plot = (nng.plots_spatial.SpatialPlot(plot_topo=False)
            .plot_ratio(cur_res_df, "residual", cb_label=f"{im2}", cmap_limits=(-1.0, 1.0, 0.1))
            .plot_sites(cur_sim_im_df)
            .plot_sites(cur_sim_im_df.loc[cur_sim_im_df.index.isin(bad_records.index)], style="t0.1c", fill="blue")
            .plot_faults([cur_event_id], pen="0.75p,red")
            .plot_hypocentre(cur_rel_info.hypo_lon, cur_rel_info.hypo_lat)
    )

    fig = nng.plot_utils.pygmt_side_by_side(res_1_plot.fig, res_2_plot.fig, dpi=300, title_1=f"{im1} Residuals", title_2=f"{im2} Residuals")  
    fig.tight_layout()
    plt.show(fig)

## Indidivual Record

In [ ]:
cur_record = cur_record_loss.sort_values("mean_loss", ascending=False).iloc[0]


In [ ]:
print("Record Int ID:", cur_record.name)
print("Sample Weight:", cur_record.sample_weight)
print("Mean Loss:", cur_record.mean_loss)

In [ ]:
cur_joined = cur_sim_im_df.loc[cur_record.name, run_config.ims].to_frame("sim").astype(float)
cur_joined["pred_mean"] = cur_pred_im_df.loc[cur_record.name, run_config.pred_mean_keys].values.astype(float)
cur_joined["pred_std"] = cur_pred_im_df.loc[cur_record.name, run_config.pred_std_keys].values.astype(float)

cur_joined["ln_sim"] = np.log(cur_joined["sim"])
cur_joined["ln_pred_mean"] = np.log(cur_joined["pred_mean"])
cur_joined["ln_residual"] = cur_joined["ln_sim"] - cur_joined["ln_pred_mean"]

cur_joined["squared_error"] = cur_joined["ln_residual"] ** 2

cur_joined["gaussian_nll"] = 0.5 * (np.log(2 * cur_joined["pred_std"] ** 2) + (cur_joined["squared_error"] / (cur_joined["pred_std"] ** 2)))

In [ ]:
cur_joined

## Mean Realisations Loss

In [ ]:
rel_loss= record_loss.groupby("rel_int_id")["mean_loss"].mean().sort_values(ascending=False).to_frame("loss")
rel_loss["rel_id"] = rel_df.loc[rel_loss.index, "rel_id"].values
rel_loss["magnitude"] = rel_df.loc[rel_loss.index, "magnitude"].values
rel_loss["event_id"] = rel_df.loc[rel_loss.index, "event_id"].values

rel_loss = rel_loss.set_index("rel_id")

### Magnitude

In [ ]:
table_1 = rel_loss.sort_values("magnitude", ascending=False).head(25).to_html()
table_2 = rel_loss.sort_values("loss", ascending=False).iloc[25:50].to_html()

display_html(f"""
    <div style="display:flex; gap:20px;">
        <div>{table_1}</div>
        <div>{table_2}</div>
    </div>
""", raw=True)

## Event Magnitude vs Loss

In [ ]:
cur_events = event_df.loc[val_df.event_int_id.unique()].sort_values("magnitude", ascending=False).event_id.values[:10].astype(str)

In [ ]:
sns.catplot(data=rel_loss.loc[rel_loss.event_id.isin(cur_events)].astype({"event_id": str}), x="event_id", y="loss", height=8, aspect=2, order=cur_events[::-1])

Increasing magnitude from left to right